# 7-class subcellular localization — retrained for maximum accuracy

The 10-class model never predicts three compartments (Golgi apparatus, Lysosome/Vacuole,
Peroxisome) — each has too few examples. Here we **remove them and retrain from scratch**, so the
model spends all of its capacity on the seven compartments that have enough data.

Every accuracy lever is on:
1. **ESM-2 650M** encoder with **attention pooling** over residues.
2. **1024-residue context, dual-end truncation** (keeps both termini).
3. **Plain cross-entropy** — with the rare classes gone, all 7 classes are well represented,
   so CE maximizes accuracy without starving anything (no focal-loss trade-off needed).
4. **Layer-wise LR decay**, cosine schedule, 4 epochs.
5. **Test-time augmentation** over 3 truncation windows.

> Set `MAX_ACCURACY = False` in the config cell for a ~25-minute 150M run instead of ~1.5–2 h.

> Honest note: this task is easier than the 10-class problem *by construction* — the hardest
> classes were removed. It is not comparable to a 10-class number.

## 1 · Setup

In [ ]:
# Kaggle/Colab ship a GPU-matched PyTorch. PIN it so upgrading libs can't swap in a
# build without kernels for this GPU ('no kernel image available').
import torch; _T = torch.__version__
!pip -q install -U transformers datasets accelerate umap-learn "torch=={_T}"
print('torch', torch.__version__, '| cuda', torch.version.cuda)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use ONE gpu (Kaggle free tier can be 2x T4)
import torch, numpy as np, pandas as pd
assert torch.cuda.is_available(), "No GPU. Kaggle: Settings -> Accelerator -> GPU; also set Internet -> On."
device='cuda'; print("GPU:", torch.cuda.get_device_name(0))

## 2 · Config

In [ ]:
MAX_ACCURACY = True     # True -> 650M / 1024 ctx / 4 epochs / TTA   (~1.5-2 h on a T4)
                        # False -> 150M / 512 ctx / 3 epochs         (~25 min)

if MAX_ACCURACY:
    MODEL_NAME='facebook/esm2_t33_650M_UR50D'; MAX_LENGTH=1024; EPOCHS=4
    BATCH=2; ACCUM=16; CKPT=True; TTA=['dual','head','tail']
else:
    MODEL_NAME='facebook/esm2_t30_150M_UR50D'; MAX_LENGTH=512;  EPOCHS=3
    BATCH=8; ACCUM=2;  CKPT=False; TTA=['dual']
print(MODEL_NAME, '| ctx', MAX_LENGTH, '| epochs', EPOCHS, '| tta', TTA)

## 3 · Load DeepLoc and drop the three rare compartments

In [ ]:
from datasets import load_dataset, Dataset
DROP={'Golgi.apparatus','Lysosome/Vacuole','Peroxisome'}
raw=load_dataset('AI4Protein/DeepLocMulti')
def frame(sp):
    d=raw[sp].to_pandas()
    d['label']=d['location'].str.split(',').str[0]
    d=d[~d['label'].isin(DROP)]
    return d[['aa_seq','label']].rename(columns={'aa_seq':'sequence'}).dropna().drop_duplicates('sequence')
train,val,test=frame('train'),frame('validation'),frame('test')
print(f'{len(train)} train / {len(val)} val / {len(test)} test  |  {train["label"].nunique()} classes')
print(train['label'].value_counts().to_dict())

## 4 · Tokenize (dual-end truncation; extra windows for TTA)

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding
labels=sorted(train['label'].unique()); label2id={l:i for i,l in enumerate(labels)}
id2label={i:l for l,i in label2id.items()}; num_labels=len(labels)
tok=AutoTokenizer.from_pretrained(MODEL_NAME)
def trunc(s,n,mode):
    if len(s)<=n: return s
    if mode=='head': return s[:n]
    if mode=='tail': return s[-n:]
    h=n//2; return s[:h]+s[-(n-h):]
def to_ds(df,mode='dual'):
    ds=Dataset.from_pandas(df.reset_index(drop=True))
    def f(b):
        t=tok([trunc(s,MAX_LENGTH-2,mode) for s in b['sequence']],truncation=True,max_length=MAX_LENGTH)
        t['labels']=[label2id[l] for l in b['label']]; return t
    return ds.map(f,batched=True,remove_columns=['sequence','label'])
train_ds=to_ds(train); val_ds=to_ds(val); collator=DataCollatorWithPadding(tok)

## 5 · Model: ESM-2 + attention pooling (plain cross-entropy)

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from transformers import AutoModel
class EsmAttentionClassifier(nn.Module):
    def __init__(self, name, num_labels, dropout=0.1):
        super().__init__()
        self.esm=AutoModel.from_pretrained(name); h=self.esm.config.hidden_size
        self.attn=nn.Linear(h,1); self.dropout=nn.Dropout(dropout)
        self.classifier=nn.Sequential(nn.Linear(2*h,h),nn.GELU(),nn.Dropout(dropout),nn.Linear(h,num_labels))
        self.config=self.esm.config; self.config.num_labels=num_labels
    def gradient_checkpointing_enable(self,**k): self.esm.gradient_checkpointing_enable(**k)
    def forward(self, input_ids=None, attention_mask=None, labels=None, **kw):
        H=self.esm(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state
        pad=(attention_mask==0).unsqueeze(-1)
        w=torch.softmax(self.attn(H).masked_fill(pad,float('-inf')),dim=1)
        feat=self.dropout(torch.cat([(w*H).sum(1), H.masked_fill(pad,float('-inf')).max(1).values],-1))
        logits=self.classifier(feat)
        loss=None if labels is None else F.cross_entropy(logits,labels)
        return {'loss':loss,'logits':logits}

def llrd_groups(model, base_lr, wd=0.01, decay=0.9):
    layers=model.esm.encoder.layer; n=len(layers)
    head=[p for nm,p in model.named_parameters() if p.requires_grad and not nm.startswith('esm.encoder.layer')]
    groups=[{'params':head,'lr':base_lr,'weight_decay':wd}]
    for i,l in enumerate(layers):
        groups.append({'params':[p for p in l.parameters() if p.requires_grad],'lr':base_lr*decay**(n-1-i),'weight_decay':wd})
    return groups

## 6 · Train

If you hit CUDA OOM: set `BATCH=1`, `ACCUM=32` in the config cell, then Restart & run all.

In [ ]:
from transformers import TrainingArguments, Trainer, set_seed
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
set_seed(42)
def compute_metrics(p):
    pr=np.argmax(p.predictions,-1)
    return {'accuracy':accuracy_score(p.label_ids,pr),'macro_f1':f1_score(p.label_ids,pr,average='macro'),'mcc':matthews_corrcoef(p.label_ids,pr)}

model=EsmAttentionClassifier(MODEL_NAME,num_labels).cuda()
if CKPT: model.gradient_checkpointing_enable(); model.esm.config.use_cache=False
args=TrainingArguments(output_dir='out7', num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=max(4,BATCH*2), gradient_accumulation_steps=ACCUM,
    learning_rate=2e-5, lr_scheduler_type='cosine', warmup_ratio=0.1, weight_decay=0.01,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='accuracy', greater_is_better=True, save_total_limit=1,
    fp16=True, logging_steps=20, report_to='none', seed=42)
opt=torch.optim.AdamW(llrd_groups(model,2e-5))
trainer=Trainer(model=model,args=args,train_dataset=train_ds,eval_dataset=val_ds,
    compute_metrics=compute_metrics,data_collator=collator,optimizers=(opt,None))
trainer.train()

## 7 · Evaluate with test-time augmentation

In [ ]:
def softmax(x): e=np.exp(x-x.max(1,keepdims=True)); return e/e.sum(1,keepdims=True)
probs=np.mean([softmax(trainer.predict(to_ds(test,m)).predictions) for m in TTA],axis=0)
y_true=np.array([label2id[l] for l in test['label']]); y_pred=probs.argmax(1)
print('averaged %d TTA window(s)'%len(TTA))
print('TEST accuracy %.4f | macro-F1 %.4f | MCC %.4f' % (accuracy_score(y_true,y_pred),
      f1_score(y_true,y_pred,average='macro'), matthews_corrcoef(y_true,y_pred)))
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
print(classification_report(y_true,y_pred,target_names=labels,zero_division=0))

In [ ]:
import seaborn as sns, matplotlib.pyplot as plt; sns.set_theme(style='whitegrid')
cm=confusion_matrix(y_true,y_pred,normalize='true')
fig,ax=plt.subplots(figsize=(8,7))
ConfusionMatrixDisplay(cm,display_labels=labels).plot(ax=ax,xticks_rotation=45,values_format='.2f',cmap='Blues',colorbar=True)
ax.set_title('Confusion Matrix (row-normalized) — 7-class test'); plt.tight_layout(); plt.show()

## 8 · Embedding map

In [ ]:
import umap
@torch.no_grad()
def embed(seqs,bs=8):
    model.eval(); v=[]
    for i in range(0,len(seqs),bs):
        s=[trunc(x,MAX_LENGTH-2,'dual') for x in seqs[i:i+bs]]
        b=tok(s,return_tensors='pt',truncation=True,padding=True,max_length=MAX_LENGTH).to('cuda')
        h=model.esm(**b).last_hidden_state; m=b['attention_mask'].unsqueeze(-1).float()
        v.append(((h*m).sum(1)/m.sum(1).clamp(min=1)).float().cpu().numpy())
    return np.concatenate(v)
emb=embed(test['sequence'].tolist())
xy=umap.UMAP(n_neighbors=25,min_dist=0.3,metric='cosine',random_state=42).fit_transform(emb)
plt.figure(figsize=(11,8.5))
for c,col in zip(labels,sns.color_palette('tab10',len(labels))):
    m=test['label'].values==c; plt.scatter(xy[m,0],xy[m,1],s=10,alpha=0.6,color=col,label=c,linewidths=0)
plt.legend(markerscale=2,fontsize=9); plt.title('Retrained 7-class ESM-2 embeddings (UMAP)'); plt.tight_layout(); plt.show()

## 9 · Export results.js for the dashboard

Download the file and send it over — it drops straight into the dashboard as the 7-class view.

In [ ]:
import json
from sklearn.metrics import precision_recall_fscore_support
conf=probs.max(1); ids=list(range(len(labels)))
met={'accuracy':float(accuracy_score(y_true,y_pred)),'macro_f1':float(f1_score(y_true,y_pred,average='macro')),
     'weighted_f1':float(f1_score(y_true,y_pred,average='weighted')),'mcc':float(matthews_corrcoef(y_true,y_pred)),'n_test':int(len(y_true))}
pr,rc,f1c,sup=precision_recall_fscore_support(y_true,y_pred,labels=ids,zero_division=0)
per_class=[{'label':labels[i],'precision':float(pr[i]),'recall':float(rc[i]),'f1':float(f1c[i]),'support':int(sup[i])} for i in ids]
cmc=confusion_matrix(y_true,y_pred,labels=ids); cmn=cmc/cmc.sum(1,keepdims=True).clip(min=1)
tr=train['label'].value_counts().to_dict(); distribution=[{'label':c,'count':int(tr.get(c,0))} for c in labels]
points=[{'x':round(float(xy[i,0]),3),'y':round(float(xy[i,1]),3),'t':int(y_true[i]),'p':int(probs[i].argmax()),'c':round(float(conf[i]),3)} for i in range(len(y_true))]
seqs=test['sequence'].tolist(); examples=[]
for ci,c in enumerate(labels):
    mask=(y_true==ci)&(y_pred==ci)
    if not mask.any(): continue
    idx=np.where(mask)[0]; best=idx[conf[idx].argmax()]; order=probs[best].argsort()[::-1][:3]
    examples.append({'true':c,'length':len(seqs[best]),'preview':seqs[best][:60],'top3':[{'label':labels[j],'prob':round(float(probs[best,j]),3)} for j in order]})
src=f'retrained 7-class ESM-2 {"650M" if MAX_ACCURACY else "150M"} + attention pooling'+(' + TTA' if len(TTA)>1 else '')
res={'model':MODEL_NAME,'source':src,
  'dataset':{'name':'DeepLocMulti — 7 well-represented compartments','n_train':int(len(train)),'n_test':int(len(y_true)),'n_classes':len(labels)},
  'classes':labels,'metrics':met,'per_class':per_class,
  'confusion':{'counts':cmc.astype(int).tolist(),'normalized':np.round(cmn,3).tolist()},
  'distribution':distribution,'points':points,'examples':examples}
open('results_7class.js','w').write('window.RESULTS_7CLASS = '+json.dumps(res,separators=(',',':'))+';\n')
print('results_7class.js written — accuracy %.4f' % met['accuracy'])
# Download — works on Colab; on Kaggle it lands in /kaggle/working
try:
    from google.colab import files; files.download('results_7class.js')
except Exception:
    print('Saved results_7class.js to the working directory — download it from the Files/Output panel.')

---
Model: ESM-2 (Lin 2023) · Data: DeepLoc (Almagro Armenteros 2017) · Pooling: Light Attention (Stärk 2021)